# Notebook 2: Tokenization & Sampling From Scratch

**Goal:** Implement BPE tokenization and all major sampling strategies. Mistral's live coding round specifically asks for top-k and top-p from scratch.

**Topics:**
1. Character-level tokenizer (baseline)
2. Byte-Pair Encoding (BPE) — simplified implementation
3. Temperature scaling
4. Greedy decoding
5. Top-k sampling
6. Top-p (nucleus) sampling
7. Min-p sampling
8. Repetition penalty
9. Beam search
10. Full autoregressive generation loop

In [ ]:
import numpy as np
import math
from typing import List, Dict, Tuple, Optional
from collections import Counter, defaultdict

np.random.seed(42)

---
## 1. Character-Level Tokenizer (Warm-Up)

Simplest possible tokenizer: each character = one token.

In [ ]:
class CharTokenizer:
    def __init__(self, text: str):
        # Build vocabulary from all unique chars in text
        chars = sorted(set(text))
        self.vocab: Dict[str, int] = {ch: i for i, ch in enumerate(chars)}
        self.inv_vocab: Dict[int, str] = {i: ch for ch, i in self.vocab.items()}
        self.vocab_size = len(chars)
    
    def encode(self, text: str) -> List[int]:
        """Text -> list of token ids"""
        # YOUR CODE HERE
        raise NotImplementedError()
    
    def decode(self, ids: List[int]) -> str:
        """List of token ids -> text"""
        # YOUR CODE HERE
        raise NotImplementedError()

sample_text = "hello world! the quick brown fox jumps over the lazy dog."
tok = CharTokenizer(sample_text)
print(f'Vocab size: {tok.vocab_size}')
# ids = tok.encode("hello")
# print('Encoded:', ids)
# print('Decoded:', tok.decode(ids))

In [ ]:
# REFERENCE
class CharTokenizerRef:
    def __init__(self, text: str):
        chars = sorted(set(text))
        self.vocab = {ch: i for i, ch in enumerate(chars)}
        self.inv_vocab = {i: ch for ch, i in self.vocab.items()}
        self.vocab_size = len(chars)
    
    def encode(self, text: str) -> List[int]:
        return [self.vocab[ch] for ch in text if ch in self.vocab]
    
    def decode(self, ids: List[int]) -> str:
        return ''.join(self.inv_vocab[i] for i in ids)

CharTokenizer = CharTokenizerRef
tok = CharTokenizer(sample_text)
ids = tok.encode("hello")
print('Encoded:', ids)
print('Decoded:', tok.decode(ids))
print('Vocab size:', tok.vocab_size)

---
## 2. Byte-Pair Encoding (BPE) — Simplified

**Algorithm:**
1. Start with character-level vocabulary
2. Repeat N times:
   a. Find the most frequent consecutive pair of tokens
   b. Merge that pair into a new token
   c. Replace all occurrences of the pair in the corpus

**Interview Q:** *Why does BPE work? What's the tradeoff with vocabulary size?*

Answer: BPE balances sequence length vs vocab size. Larger vocab → shorter sequences (faster inference) but more memory for embeddings. Small vocab (chars) → very long sequences. ~32k-128k is the sweet spot.

In [ ]:
def get_pairs(word_freqs: Dict[Tuple, int]) -> Counter:
    """
    Count all consecutive pair frequencies across all words.
    
    Args:
        word_freqs: maps tuple of token strings to frequency
        e.g., {('h','e','l','l','o'): 5, ('w','o','r','l','d'): 3}
    Returns:
        Counter of pair -> total frequency
    """
    # YOUR CODE HERE
    raise NotImplementedError()


def merge_pair(pair: Tuple[str, str], word_freqs: Dict[Tuple, int]) -> Dict[Tuple, int]:
    """
    Merge all occurrences of pair in word_freqs.
    e.g., pair=('h','e'), word ('h','e','l','l','o') -> ('he','l','l','o')
    """
    # YOUR CODE HERE
    raise NotImplementedError()


def train_bpe(corpus: str, num_merges: int) -> Tuple[Dict[Tuple, int], List[Tuple[str, str]]]:
    """
    Train BPE on a corpus string.
    
    Returns:
        word_freqs: final vocabulary of word tuples
        merges: ordered list of merge rules
    """
    # Step 1: Initialize - split each word into chars, add end-of-word marker
    words = corpus.lower().split()
    # Count word frequencies
    word_count: Counter = Counter(words)
    # Represent each word as tuple of chars + </w> end marker
    word_freqs: Dict[Tuple, int] = {}
    for word, freq in word_count.items():
        char_tuple = tuple(list(word) + ['</w>'])
        word_freqs[char_tuple] = freq
    
    merges: List[Tuple[str, str]] = []
    
    # Step 2: Repeat num_merges times
    for _ in range(num_merges):
        # YOUR CODE HERE
        # a. Get all pairs and their frequencies
        # b. Find the most common pair
        # c. Merge it and record the merge rule
        pass
    
    return word_freqs, merges


corpus = "low lower newest widest"
# word_freqs, merges = train_bpe(corpus, num_merges=10)
# print('Learned merges:')
# for i, m in enumerate(merges):
#     print(f'  {i+1}. {m[0]} + {m[1]} -> {m[0]+m[1]}')

In [ ]:
# REFERENCE
def get_pairs_ref(word_freqs: Dict[Tuple, int]) -> Counter:
    pairs: Counter = Counter()
    for word_tuple, freq in word_freqs.items():
        for i in range(len(word_tuple) - 1):
            pairs[(word_tuple[i], word_tuple[i+1])] += freq
    return pairs


def merge_pair_ref(pair: Tuple[str, str], word_freqs: Dict[Tuple, int]) -> Dict[Tuple, int]:
    new_word_freqs = {}
    merged = ''.join(pair)
    for word_tuple, freq in word_freqs.items():
        new_word = []
        i = 0
        while i < len(word_tuple):
            if i < len(word_tuple) - 1 and word_tuple[i] == pair[0] and word_tuple[i+1] == pair[1]:
                new_word.append(merged)
                i += 2
            else:
                new_word.append(word_tuple[i])
                i += 1
        new_word_freqs[tuple(new_word)] = freq
    return new_word_freqs


def train_bpe_ref(corpus: str, num_merges: int):
    words = corpus.lower().split()
    word_count = Counter(words)
    word_freqs = {tuple(list(w) + ['</w>']): freq for w, freq in word_count.items()}
    merges = []
    
    for _ in range(num_merges):
        pairs = get_pairs_ref(word_freqs)
        if not pairs:
            break
        best_pair = pairs.most_common(1)[0][0]
        word_freqs = merge_pair_ref(best_pair, word_freqs)
        merges.append(best_pair)
    
    return word_freqs, merges


get_pairs = get_pairs_ref
merge_pair = merge_pair_ref
train_bpe = train_bpe_ref

corpus = "low lower lowest newest widest widest widest"
word_freqs, merges = train_bpe(corpus, num_merges=10)
print('Learned merges:')
for i, m in enumerate(merges):
    print(f'  {i+1}. "{m[0]}" + "{m[1]}" -> "{m[0]+m[1]}"')

print('\nFinal word representations:')
for wt, freq in word_freqs.items():
    print(f'  {" | ".join(wt)} (freq={freq})')

---
## 3. Temperature Scaling

**Temperature T** controls the sharpness of the distribution.
- T < 1: more peaked (model is more confident, less diverse)
- T = 1: original distribution
- T > 1: flatter (more uniform, more diverse/random)

**Formula:** Divide logits by T BEFORE softmax: `softmax(logits / T)`

In [ ]:
def apply_temperature(logits: np.ndarray, temperature: float) -> np.ndarray:
    """
    Apply temperature scaling to logits.
    temperature > 0 required.
    
    Returns: scaled logits (NOT probabilities — do not apply softmax here)
    """
    if temperature <= 0:
        raise ValueError(f'Temperature must be > 0, got {temperature}')
    # YOUR CODE HERE
    raise NotImplementedError()


# Visualize how temperature affects the distribution
def softmax(x):
    x = x - x.max()
    e = np.exp(x)
    return e / e.sum()

logits = np.array([2.0, 1.0, 0.5, 0.1, -0.5])
print('Vocab: ["the", "a", "cat", "dog", "ran"]')

# for temp in [0.1, 0.5, 1.0, 2.0]:
#     probs = softmax(apply_temperature(logits, temp))
#     print(f'T={temp}: {probs.round(3)}')

In [ ]:
# REFERENCE + visualization
def apply_temperature_ref(logits: np.ndarray, temperature: float) -> np.ndarray:
    if temperature <= 0:
        raise ValueError(f'Temperature must be > 0, got {temperature}')
    return logits / temperature

apply_temperature = apply_temperature_ref

logits = np.array([2.0, 1.0, 0.5, 0.1, -0.5])
labels = ['"the"', '"a"', '"cat"', '"dog"', '"ran"']

print('Token probabilities at different temperatures:')
print(f'{"Token":<10}', end='')
for temp in [0.1, 0.5, 1.0, 2.0]:
    print(f'T={temp:<6}', end='')
print()
probs_by_temp = {t: softmax(apply_temperature(logits, t)) for t in [0.1, 0.5, 1.0, 2.0]}
for i, label in enumerate(labels):
    print(f'{label:<10}', end='')
    for t in [0.1, 0.5, 1.0, 2.0]:
        print(f'{probs_by_temp[t][i]:.3f}   ', end='')
    print()

---
## 4. Greedy Decoding

Simply pick the highest probability token at each step. Fast but often produces repetitive, low-quality text.

In [ ]:
def greedy_sample(logits: np.ndarray) -> int:
    """
    Returns the argmax token id from logits.
    logits: (vocab_size,) — raw model output
    """
    # YOUR CODE HERE
    raise NotImplementedError()


logits = np.array([0.1, 2.5, 0.8, 1.2, 0.3])
# print('Greedy choice:', greedy_sample(logits))  # should be 1

In [ ]:
# REFERENCE
def greedy_sample_ref(logits: np.ndarray) -> int:
    return int(np.argmax(logits))

greedy_sample = greedy_sample_ref
logits = np.array([0.1, 2.5, 0.8, 1.2, 0.3])
print('Greedy choice:', greedy_sample(logits))  # 1

---
## 5. Top-k Sampling ★ (Asked in Mistral Coding Round)

**Algorithm:**
1. Keep only the top-k logits
2. Set all others to -inf
3. Apply softmax to get probabilities over just those k tokens
4. Sample from that distribution

**Why top-k?** Prevents sampling very low-probability 'tail' tokens that degrade quality.

In [ ]:
def top_k_sample(logits: np.ndarray, k: int, temperature: float = 1.0) -> int:
    """
    Top-k sampling.
    
    Args:
        logits: (vocab_size,) raw model logits
        k: number of top tokens to consider
        temperature: applied before filtering
    Returns:
        sampled token id
    """
    # YOUR CODE HERE
    # Step 1: Apply temperature
    # Step 2: Find top-k indices and values
    # Step 3: Create filtered logits (everything else = -inf)
    # Step 4: Softmax
    # Step 5: Sample from the resulting distribution
    # Hint: np.random.choice(vocab_size, p=probs)
    raise NotImplementedError()


# Test: should always sample from top 3 tokens
np.random.seed(0)
logits = np.array([5.0, 4.0, 3.0, 0.1, 0.1, 0.1])  # tokens 0, 1, 2 dominate
samples = [top_k_sample(logits, k=3) for _ in range(100)]
# All samples should be in {0, 1, 2}
# print('Unique tokens sampled:', set(samples))  # should be subset of {0, 1, 2}

In [ ]:
# REFERENCE
def top_k_sample_ref(logits: np.ndarray, k: int, temperature: float = 1.0) -> int:
    logits = apply_temperature(logits.astype(float), temperature)
    
    # Get indices of top-k elements (unsorted)
    top_k_indices = np.argpartition(logits, -k)[-k:]  # faster than full sort
    
    # Create -inf mask
    filtered_logits = np.full_like(logits, -np.inf)
    filtered_logits[top_k_indices] = logits[top_k_indices]
    
    probs = softmax(filtered_logits)
    return int(np.random.choice(len(probs), p=probs))


top_k_sample = top_k_sample_ref

np.random.seed(0)
logits = np.array([5.0, 4.0, 3.0, 0.1, 0.1, 0.1])
samples = [top_k_sample(logits, k=3) for _ in range(200)]
from collections import Counter as C
print('Top-k=3 sampling distribution:', dict(C(samples)))
assert max(samples) <= 2, 'Should only sample top-3 tokens'

---
## 6. Top-p (Nucleus) Sampling ★ (Asked in Mistral Coding Round)

**Algorithm:**
1. Sort tokens by probability (highest first)
2. Accumulate probabilities until the cumulative sum >= p
3. Keep only those tokens (the 'nucleus')
4. Renormalize and sample

**Why top-p over top-k?**  
The nucleus size is adaptive — in high-uncertainty situations (flat distribution), many tokens are included. In low-uncertainty (peaked distribution), fewer tokens are included. Top-k always uses exactly k, regardless of the distribution shape.

In [ ]:
def top_p_sample(logits: np.ndarray, p: float, temperature: float = 1.0) -> int:
    """
    Top-p (nucleus) sampling.
    
    Args:
        logits: (vocab_size,) raw model logits
        p: cumulative probability threshold (e.g., 0.9)
        temperature: applied before filtering
    Returns:
        sampled token id
    """
    assert 0 < p <= 1.0, 'p must be in (0, 1]'
    
    # YOUR CODE HERE
    # Step 1: Apply temperature, compute probabilities
    # Step 2: Sort by probability (descending)
    # Step 3: Compute cumulative probabilities
    # Step 4: Find where cumulative >= p, mask the rest out
    # Step 5: Renormalize and sample
    # IMPORTANT: Always keep at least 1 token (the top-1)
    raise NotImplementedError()


# Test with a very peaked distribution (only 1 token should be in nucleus)
logits_peaked = np.array([100.0, 0.0, 0.0, 0.0, 0.0])
np.random.seed(0)
samples = [top_p_sample(logits_peaked, p=0.9) for _ in range(100)]
# print('Peaked (should be all 0):', set(samples))

# Test with uniform distribution (all tokens in nucleus for p=0.9)
logits_flat = np.zeros(10)
samples_flat = [top_p_sample(logits_flat, p=0.9) for _ in range(200)]
# print('Flat (should span all 10):', set(samples_flat))

In [ ]:
# REFERENCE
def top_p_sample_ref(logits: np.ndarray, p: float, temperature: float = 1.0) -> int:
    assert 0 < p <= 1.0
    
    logits = apply_temperature(logits.astype(float), temperature)
    probs = softmax(logits)
    
    # Sort descending by probability
    sorted_indices = np.argsort(probs)[::-1]
    sorted_probs = probs[sorted_indices]
    
    # Cumulative sum
    cumulative_probs = np.cumsum(sorted_probs)
    
    # Find cutoff: first index where cumulative >= p
    # We include this token (shift by 1 so we keep at least the top token)
    cutoff_idx = np.searchsorted(cumulative_probs, p, side='left') + 1
    cutoff_idx = max(1, min(cutoff_idx, len(probs)))  # at least 1 token
    
    nucleus_indices = sorted_indices[:cutoff_idx]
    nucleus_probs = probs[nucleus_indices]
    nucleus_probs = nucleus_probs / nucleus_probs.sum()  # renormalize
    
    return int(np.random.choice(nucleus_indices, p=nucleus_probs))


top_p_sample = top_p_sample_ref

np.random.seed(0)
logits_peaked = np.array([100.0, 0.0, 0.0, 0.0, 0.0])
samples = [top_p_sample(logits_peaked, p=0.9) for _ in range(20)]
print('Peaked (p=0.9):', set(samples))  # should be {0}

logits_flat = np.zeros(10)
samples_flat = [top_p_sample(logits_flat, p=0.9) for _ in range(200)]
print('Flat (p=0.9), tokens sampled:', sorted(set(samples_flat)))  # should span many tokens

# Mixed
logits_mixed = np.array([3.0, 2.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0])
samples_mixed = [top_p_sample(logits_mixed, p=0.9) for _ in range(500)]
print('Mixed (p=0.9):', dict(C(samples_mixed)))

---
## 7. Min-p Sampling

**Newer technique:** Keep tokens whose probability is at least `min_p * p_max` where `p_max` is the max token probability.

**Why?** Scales the threshold relative to how confident the model is, avoiding the fixed-cutoff problem of top-k.

In [ ]:
def min_p_sample(logits: np.ndarray, min_p: float, temperature: float = 1.0) -> int:
    """
    Min-p sampling: keep tokens with prob >= min_p * max_prob.
    """
    # YOUR CODE HERE
    raise NotImplementedError()

# REFERENCE
def min_p_sample_ref(logits: np.ndarray, min_p: float, temperature: float = 1.0) -> int:
    logits = apply_temperature(logits.astype(float), temperature)
    probs = softmax(logits)
    threshold = min_p * probs.max()
    filtered_probs = np.where(probs >= threshold, probs, 0.0)
    filtered_probs = filtered_probs / filtered_probs.sum()
    return int(np.random.choice(len(probs), p=filtered_probs))

min_p_sample = min_p_sample_ref

logits = np.array([3.0, 2.5, 2.0, 0.1, 0.1])
np.random.seed(0)
samples = [min_p_sample(logits, min_p=0.1) for _ in range(300)]
print('Min-p=0.1 distribution:', dict(C(samples)))

---
## 8. Repetition Penalty

Penalize tokens that have already appeared to reduce repetition.

**Formula:** `logit[t] = logit[t] / penalty if logit[t] > 0 else logit[t] * penalty`

penalty > 1.0 reduces probability of repeated tokens.

In [ ]:
def apply_repetition_penalty(
    logits: np.ndarray,
    generated_tokens: List[int],
    penalty: float
) -> np.ndarray:
    """
    Apply repetition penalty to logits for tokens already generated.
    
    For each token id in generated_tokens:
        if logit > 0: logit /= penalty
        else:         logit *= penalty
    """
    # YOUR CODE HERE
    raise NotImplementedError()


# REFERENCE
def apply_repetition_penalty_ref(
    logits: np.ndarray,
    generated_tokens: List[int],
    penalty: float
) -> np.ndarray:
    logits = logits.copy().astype(float)
    for token_id in set(generated_tokens):
        if logits[token_id] > 0:
            logits[token_id] /= penalty
        else:
            logits[token_id] *= penalty
    return logits

apply_repetition_penalty = apply_repetition_penalty_ref

logits = np.array([2.0, 1.5, 1.0, 0.5])
generated = [0, 1]  # tokens 0 and 1 already appeared
penalized = apply_repetition_penalty(logits, generated, penalty=1.5)
print('Original logits: ', logits)
print('Penalized logits:', penalized.round(3))

---
## 9. Putting It All Together: Generation Loop

A realistic autoregressive generation function with all sampling strategies.

In [ ]:
class MockLanguageModel:
    """Fake LM for testing our sampling code without a real model."""
    def __init__(self, vocab_size: int = 20):
        self.vocab_size = vocab_size
        np.random.seed(123)
        # Pre-generate some "transition" probabilities for fake 'coherence'
        self.transition = np.random.dirichlet(np.ones(vocab_size), size=vocab_size)
    
    def get_logits(self, token_ids: List[int]) -> np.ndarray:
        """Given context, return logits for next token. (batch_size=1)"""
        if not token_ids:
            return np.random.randn(self.vocab_size)
        last_token = token_ids[-1] % self.vocab_size
        # Add some randomness
        logits = np.log(self.transition[last_token] + 1e-8) + np.random.randn(self.vocab_size) * 0.3
        return logits


def generate(
    model: MockLanguageModel,
    prompt_ids: List[int],
    max_new_tokens: int = 20,
    temperature: float = 1.0,
    top_k: Optional[int] = None,
    top_p: Optional[float] = None,
    repetition_penalty: float = 1.0,
    eos_token_id: Optional[int] = None,
) -> List[int]:
    """
    Autoregressive generation with full sampling pipeline.
    
    Returns: list of generated token ids (not including prompt)
    """
    # YOUR CODE HERE
    # For each step up to max_new_tokens:
    #   1. Get logits from model
    #   2. Apply repetition penalty if penalty != 1.0
    #   3. If top_k set: use top_k_sample, else if top_p set: use top_p_sample, else greedy
    #   4. Apply temperature when calling sampling functions
    #   5. Append to context
    #   6. Stop if eos_token_id generated
    raise NotImplementedError()


# model = MockLanguageModel(vocab_size=20)
# generated = generate(model, prompt_ids=[5, 3], max_new_tokens=15, temperature=0.8, top_p=0.9)
# print('Generated tokens:', generated)

In [ ]:
# REFERENCE
def generate_ref(
    model: MockLanguageModel,
    prompt_ids: List[int],
    max_new_tokens: int = 20,
    temperature: float = 1.0,
    top_k: Optional[int] = None,
    top_p: Optional[float] = None,
    repetition_penalty: float = 1.0,
    eos_token_id: Optional[int] = None,
) -> List[int]:
    context = list(prompt_ids)
    generated = []
    
    for _ in range(max_new_tokens):
        logits = model.get_logits(context)
        
        if repetition_penalty != 1.0:
            logits = apply_repetition_penalty(logits, context + generated, repetition_penalty)
        
        if top_k is not None:
            next_token = top_k_sample(logits, k=top_k, temperature=temperature)
        elif top_p is not None:
            next_token = top_p_sample(logits, p=top_p, temperature=temperature)
        else:
            scaled = apply_temperature(logits, temperature)
            probs = softmax(scaled)
            next_token = int(np.random.choice(len(probs), p=probs))
        
        generated.append(next_token)
        context.append(next_token)
        
        if eos_token_id is not None and next_token == eos_token_id:
            break
    
    return generated


generate = generate_ref

model = MockLanguageModel(vocab_size=20)
np.random.seed(0)

print('=== Greedy ===' )
greedy = generate(model, [5, 3], max_new_tokens=10, temperature=0.01)
print('Greedy:', greedy)

print('\n=== Top-k (k=5) ===')
topk = generate(model, [5, 3], max_new_tokens=10, temperature=0.8, top_k=5)
print('Top-k:', topk)

print('\n=== Top-p (p=0.9) ===')
topp = generate(model, [5, 3], max_new_tokens=10, temperature=0.8, top_p=0.9)
print('Top-p:', topp)

---
## 10. Interview Practice Questions

Answer these out loud before moving to the next notebook:

1. **Compare top-k vs top-p:** When would you prefer each? Give an example where top-k fails.

2. **Temperature extremes:** What happens at T=0? At T=∞? What sampling strategy does T→0 converge to?

3. **BPE tradeoff:** A larger vocabulary means... (finish this sentence with 3 effects).

4. **Implement from memory:** Write top-p sampling in under 15 lines. (Practice doing it without looking.)

5. **Mistral-specific:** Why might you use a lower temperature for code generation vs. creative writing?

6. **KV cache:** Why do we cache K and V but not Q? How much memory does the KV cache use for a 7B model with 32k context?

---
**Next:** `03_mistral_api_basics.ipynb`